<a href="https://colab.research.google.com/github/marcouras/AI-engineering-fundamentals/blob/main/lezione3/Lezione3_Colab.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

---

# 🤖 AI Engineering Fundamentals
## Lezione 3 — Conversazione & Memoria

**ITS Novitas 4.0 — Sviluppatore Intelligenza Artificiale**  
Docente: Marco Uras | 📅 Martedì 26/05/2026

---

### 🎯 Obiettivi
- ✅ Gestire la conversation history (multi-turno)
- ✅ Implementare troncamento e sliding window
- ✅ Aggiungere lo streaming delle risposte
- ✅ Salvare e ricaricare la memoria su file JSON

In [4]:
# Setup
!pip install anthropic -q
from google.colab import userdata
import anthropic, os, json

os.environ["ANTHROPIC_API_KEY"] = userdata.get("ANTHROPIC_API_KEY")
client = anthropic.Anthropic()
print("✅ Setup completato!")

✅ Setup completato!


---
## 1. Conversation History — il chatbot che ricorda

Il modello **non ha memoria propria**. Per farlo 'ricordare', dobbiamo inviargli tutta la conversazione ad ogni chiamata.

In [5]:
# Chatbot multi-turno base
history = []

def chat(messaggio, system=None):
    """Invia un messaggio mantenendo la history."""
    # 1. Aggiungi il messaggio dell'utente
    history.append({"role": "user", "content": messaggio})

    # 2. Invia TUTTA la history al modello
    params = {
        "model": "claude-haiku-4-5-20251001",
        "max_tokens": 100,
        "messages": history
    }
    if system:
        params["system"] = system

    risposta = client.messages.create(**params)
    testo = risposta.content[0].text

    # 3. Aggiungi la risposta alla history
    history.append({"role": "assistant", "content": testo})

    return testo

print("✅ Funzione chat pronta!")

✅ Funzione chat pronta!


In [6]:
# Proviamo la memoria!
history = []  # Reset

print("👤 Mi chiamo Marco e sono di Sassari.")
r1 = chat("Mi chiamo Marco e sono di Sassari.")
print(f"🤖 {r1}\n")

print("👤 Qual è la capitale della Sardegna?")
r2 = chat("Qual è la capitale della Sardegna?")
print(f"🤖 {r2}\n")

print("👤 Come mi chiamo?")
r3 = chat("Come mi chiamo?")  # Ricorda il nome?
print(f"🤖 {r3}\n")

print(f"📊 Messaggi in history: {len(history)}")

👤 Mi chiamo Marco e sono di Sassari.
🤖 Piacere, Marco! Sassari è una bellissima città della Sardegna, con una storia ricca e tanto carattere. 

C'è qualcosa di specifico su cui posso aiutarti? Che si tratti di informazioni sulla tua città, domande in generale, o semplicemente una chiacchierata, sono qui per te! 😊

👤 Qual è la capitale della Sardegna?
🤖 La capitale della Sardegna è **Cagliari**, situata nella parte meridionale dell'isola.

Sassari, dove abiti tu, è invece il secondo comune più grande della Sardegna per popolazione ed è il capoluogo della provincia di Sassari, nella parte settentrionale dell'isola.

👤 Come mi chiamo?
🤖 Ti chiami **Marco**! 😊

Me l'hai detto all'inizio della nostra conversazione.

📊 Messaggi in history: 6


In [7]:
# Vediamo quanti token stiamo usando
from anthropic import Anthropic

count = client.messages.count_tokens(
    model="claude-haiku-4-5-20251001",
    messages=history
)
print(f"📊 Token nella history attuale: {count.input_tokens}")
print(f"💰 Costo stimato prossima chiamata: ${count.input_tokens / 1_000_000 * 1.0:.6f}")
print()
print("💡 Nota: la history cresce ad ogni messaggio — dobbiamo gestirla!")

📊 Token nella history attuale: 235
💰 Costo stimato prossima chiamata: $0.000235

💡 Nota: la history cresce ad ogni messaggio — dobbiamo gestirla!


---
## 2. Gestire la Context Window

Tre strategie per evitare che la history cresca all'infinito.

In [8]:
# STRATEGIA 1: Truncation
MAX_MESSAGGI = 6  # massimo 3 turni (user + assistant)

def chat_con_troncamento(messaggio, system=None):
    history.append({"role": "user", "content": messaggio})

    # Tronca se troppo lunga
    if len(history) > MAX_MESSAGGI:
        history[:] = history[-MAX_MESSAGGI:]
        print(f"  ✂️  History troncata a {MAX_MESSAGGI} messaggi")

    risposta = client.messages.create(
        model="claude-haiku-4-5-20251001",
        max_tokens=300,
        messages=history
    )
    testo = risposta.content[0].text
    history.append({"role": "assistant", "content": testo})
    return testo

# Test
history = []
for i in range(5):
    r = chat_con_troncamento(f"Messaggio numero {i+1}. Quanti messaggi ricordi?")
    print(f"Turno {i+1} | History: {len(history)} msg | Risposta: {r[:80]}...")

Turno 1 | History: 2 msg | Risposta: Ricordo questo messaggio che mi hai appena scritto. È il **messaggio numero 1** ...
Turno 2 | History: 4 msg | Risposta: Ricordo **2 messaggi**:

1. Il tuo primo messaggio dove mi chiedevi quanti messa...
Turno 3 | History: 6 msg | Risposta: Ricordo **3 messaggi**:

1. Il tuo primo messaggio ("Messaggio numero 1. Quanti ...
  ✂️  History troncata a 6 messaggi
Turno 4 | History: 7 msg | Risposta: Ricordo **4 messaggi tuoi** e **4 miei messaggi di risposta**, per un totale di ...
  ✂️  History troncata a 6 messaggi
Turno 5 | History: 7 msg | Risposta: Ricordo **5 messaggi tuoi** e **5 miei messaggi di risposta**, per un totale di ...


In [9]:
# STREAMING — output token per token
print("🌊 Risposta in streaming:\n")

full_text = ""
with client.messages.stream(
    model="claude-haiku-4-5-20251001",
    max_tokens=300,
    messages=[{"role": "user", "content": "Raccontami in 3 frasi cos'è il machine learning."}]
) as stream:
    for text in stream.text_stream:
        print(text, end="", flush=True)
        full_text += text

print(f"\n\n📊 Testo totale: {len(full_text)} caratteri")

🌊 Risposta in streaming:

# Machine Learning

Il machine learning è una branca dell'intelligenza artificiale che permette ai computer di imparare dai dati senza essere esplicitamente programmati per ogni situazione. Gli algoritmi identificano automaticamente schemi e regole nei dati, migliorando le loro prestazioni man mano che elaborano più informazioni. Viene utilizzato in molte applicazioni quotidiane, come i sistemi di raccomandazione di Netflix, i filtri anti-spam e il riconoscimento facciale.

📊 Testo totale: 477 caratteri


---
## 3. Memoria Persistente su JSON

Salviamo la history su file — il chatbot ricorda tra sessioni diverse.

In [10]:
import json, os

MEMORY_FILE = "chat_history.json"

def carica_storia():
    """Carica la history dal file JSON. Restituisce lista vuota se non esiste."""
    if os.path.exists(MEMORY_FILE):
        with open(MEMORY_FILE, "r", encoding="utf-8") as f:
            storia = json.load(f)
            print(f"📂 Storia caricata: {len(storia)} messaggi precedenti")
            return storia
    print("🆕 Nessuna storia precedente — nuova conversazione")
    return []

def salva_storia(history):
    """Salva la history su file JSON."""
    with open(MEMORY_FILE, "w", encoding="utf-8") as f:
        json.dump(history, f, ensure_ascii=False, indent=2)
    print(f"💾 Storia salvata: {len(history)} messaggi")

def chat_persistente(messaggio, system=None):
    """Chatbot con memoria persistente tra sessioni."""
    history.append({"role": "user", "content": messaggio})

    params = {
        "model": "claude-haiku-4-5-20251001",
        "max_tokens": 400,
        "messages": history
    }
    if system:
        params["system"] = system

    risposta = client.messages.create(**params)
    testo = risposta.content[0].text
    history.append({"role": "assistant", "content": testo})

    salva_storia(history)  # Salva dopo ogni messaggio
    return testo

print("✅ Funzioni di persistenza pronte!")

✅ Funzioni di persistenza pronte!


In [11]:
# Simulazione sessione 1
print("=" * 50)
print("SESSIONE 1")
print("=" * 50)

history = carica_storia()  # Carica eventuali messaggi precedenti

r = chat_persistente("Ciao! Mi chiamo Luca e studio AI engineering a Sassari.")
print(f"🤖 {r}\n")

r = chat_persistente("Qual è la lezione più difficile secondo te?")
print(f"🤖 {r}\n")

print("\n--- Fine sessione 1 ---")
print(f"History salvata con {len(history)} messaggi")

SESSIONE 1
🆕 Nessuna storia precedente — nuova conversazione
💾 Storia salvata: 2 messaggi
🤖 Ciao Luca! Piacere di conoscerti! 

Che bello, studi AI engineering a Sassari! È un campo davvero affascinante e in crescita. La Sardegna sta sviluppando anche buone realtà tech, quindi sei in un buon momento.

Come va con gli studi? Su cosa stai lavorando in questo periodo? Oppure se preferisci, posso aiutarti con progetti, domande tecniche o altro - sono qui per quello! 😊

💾 Storia salvata: 4 messaggi
🤖 Buona domanda! Secondo me, le lezioni più "toste" in AI engineering di solito sono:

1. **Matematica alle spalle** - Algebra lineare, calcolo, probabilità e statistica sono fondamentali ma richiedono una base solida. Se non la hai, diventa frustrante dopo.

2. **Il gap tra teoria e pratica** - Capisci un algoritmo in teoria, ma implementarlo e farlo funzionare bene sono due cose diverse. Debugging di modelli è quasi un'arte.

3. **Intuizione vs formalismo** - Capire *perché* una rete neurale fa

In [12]:
# Simulazione sessione 2 — ricarica la memoria!
print("=" * 50)
print("SESSIONE 2 (nuova sessione, stessa memoria)")
print("=" * 50)

history = carica_storia()  # Ricarica dal file

r = chat_persistente("Come mi chiamo? Ricordi cosa stavo studiando?")
print(f"🤖 {r}")

SESSIONE 2 (nuova sessione, stessa memoria)
📂 Storia caricata: 4 messaggi precedenti
💾 Storia salvata: 6 messaggi
🤖 Sì, certo! Ti chiami **Luca** e studi **AI engineering a Sassari**! 😊

L'ho tenuto a mente da quando te l'hai detto all'inizio della nostra conversazione.


---
## ⭐ Esercizi

In [13]:
NOME_STUDENTE = "Lorenzo Masia"  # ← SCRIVI IL TUO NOME
if NOME_STUDENTE:
    print(f"✅ Notebook di: {NOME_STUDENTE}")
else:
    print("⚠️ Scrivi il tuo nome!")

✅ Notebook di: Lorenzo Masia


### Esercizio 1 — Chatbot multi-turno base ★☆☆
Crea un chatbot con system prompt WiData che mantiene la history. Fai almeno 4 domande collegate e verifica che risponda in modo coerente con i messaggi precedenti.

In [14]:
# ESERCIZIO 1
history = []
system_widata = """
  Sei un assistente esperto IoT che lavora nell'azienda di WiData, azienda che opera a sassari, ti occupi di smart city e sensoristica  # ← scrivi il system prompt WiData
"""

# Fai almeno 4 domande collegate
# domanda1 = chat("...", system=system_widata)
# ...
d1 = chat('Di cosa si occupa WiData?', system=system_widata)
print(d1)
print("--------------------------------")
d2 = chat("Quale é l'obbietivo di WiData?", system=system_widata)
print(d2)
print("--------------------------------")
d3 = chat("Quali sensori monta WiData?", system=system_widata)
print(d3)
print("--------------------------------")
d4 = chat("In quale cittá siete piú presenti?")
print(d4)

# WiData - Profilo Aziendale

WiData è un'azienda che opera a **Sassari** e si specializza in:

## Aree di competenza principali

🏙️ **Smart City**
- Soluzioni per città intelligenti
- Gestione urbana integrata
- Miglioramento della qualità della vita

📡 **Sensoristica IoT**
--------------------------------
# Obiettivi di WiData

Basandomi sul mio ruolo come assistente esperto IoT in WiData, gli obiettivi principali dell'azienda sono:

## 🎯 Obiettivi Strategici

**1. Digitalizzazione urbana**
- Trasformare le città tradizionali in ecosistemi intelligenti e connessi
- Implementare infrastrutture Io
--------------------------------
# Sensori WiData

Come assistente esperto di WiData, posso condividere che l'azienda si occupa di **sensoristica IoT per Smart City**, tuttavia non ho informazioni specifiche dettagliate nel mio profilo riguardo ai sensori esatti che montiamo.

## Tipologie di sensori comuni in ambito Smart City:

Generalmente, le soluzioni IoT per
----------------------------

### Esercizio 2 — Sliding Window ★★☆
Implementa la sliding window: mantieni sempre il system prompt + gli ultimi 4 turni (8 messaggi). Testa che dopo 6 turni il chatbot non ricordi più i primi messaggi ma ricordi gli ultimi.

In [15]:
# ESERCIZIO 2
MAX_TURNS = 4  # mantieni solo gli ultimi 4 turni

def chat_con_troncamento(messaggio, system=None):
    history.append({"role": "user", "content": messaggio})

    # Tronca se troppo lunga
    if len(history) > MAX_TURNS:
        history[:] = history[-MAX_TURNS:]
        print(f"  ✂️  History troncata a {MAX_TURNS} messaggi")

    risposta = client.messages.create(
        model="claude-haiku-4-5-20251001",
        max_tokens=120,
        messages=history
    )
    testo = risposta.content[0].text
    history.append({"role": "assistant", "content": testo})
    return testo

# Test: la prima informazione viene dimenticata dopo 4 turni?
history = []

# ... aggiungi altri messaggi ...
chat_con_troncamento("Mi chiamo Alice")
chat_con_troncamento("Come mi chiamo?")
chat_con_troncamento("Ho 20 anni")
chat_con_troncamento("Sono alto 1,85")
chat_con_troncamento("Ho 18 anni")
chat_con_troncamento("Mi piace la pasta")
chat_con_troncamento("Mi piace anche la carne")
chat_con_troncamento("Non mi piace nulla")
chat_con_troncamento("Come mi chiamo?")
chat_con_troncamento("Mi chiamo Lorenzo")


  ✂️  History troncata a 4 messaggi
  ✂️  History troncata a 4 messaggi
  ✂️  History troncata a 4 messaggi
  ✂️  History troncata a 4 messaggi
  ✂️  History troncata a 4 messaggi
  ✂️  History troncata a 4 messaggi
  ✂️  History troncata a 4 messaggi
  ✂️  History troncata a 4 messaggi


'Ah, scusa! 😅 Mi correggo subito:\n\n- Nome: **Lorenzo**\n- Età: 18 anni\n- Altezza: 1,85 m\n- Mi piace: la pasta, la carne\n- Non mi piace: nulla\n\nGrazie per la correzione, Lorenzo! Ora ho il nome giusto. 😊'

### Esercizio 3 — Chatbot con streaming ★★☆
Riscrivi la funzione `chat()` usando lo streaming. L'output deve apparire parola per parola nel terminale. Aggiungi anche il conteggio dei token al termine.

In [16]:
# ESERCIZIO 3
def chat_streaming(messaggio, history, system=None):
    history.append({"role": "user", "content": messaggio})

    # TODO: usa client.messages.stream() invece di client.messages.create()
    params = {
        "model": "claude-haiku-4-5-20251001",
        "max_tokens": 100,
        "messages": history
    }
    if system:
        params["system"] = system

    full_text = ""
    with client.messages.stream(**params) as stream:
        for text in stream.text_stream:
            print(text, end="", flush=True)
            # Stampa ogni token con print(text, end='', flush=True)
            full_text += text
    # Accumula il testo completo in una variabile
    # Aggiungi la risposta completa alla history
    # Alla fine stampa il numero di token usati
    history.append({"role": "assistant", "content": full_text})

message = stream.get_final_message()
print(f"Token usati: {message.usage.input_tokens + message.usage.output_tokens}")
# Test
history = []
chat_streaming("Spiegami RAG in 3 frasi", history)

Token usati: 145
# RAG (Retrieval-Augmented Generation)

**RAG** è una tecnica che combina la ricerca di informazioni da una base di dati con la generazione di testo tramite intelligenza artificiale. Quando fai una domanda, il sistema prima recupera i documenti più rilevanti dal database, poi li passa al modello AI per generare una risposta accurata e basata su fonti. In questo modo ott

### Esercizio 4 — Chatbot con memoria persistente ★★★ (Deliverable!)

Costruisci il chatbot completo `chatbot_cli.py` con:
- History multi-turno
- Sliding window (max 10 messaggi)
- Streaming
- Memoria su JSON persistente
- System prompt WiData
- Loop interattivo con `input()` (digita 'esci' per uscire)

In [23]:
# ESERCIZIO 4 — Chatbot completo (DELIVERABLE)
# Scrivi qui il codice completo del chatbot

import anthropic, json, os
from google.colab import userdata

os.environ["ANTHROPIC_API_KEY"] = userdata.get("ANTHROPIC_API_KEY")
client = anthropic.Anthropic()

MEMORY_FILE = "chatbot_widata.json"
MAX_MESSAGGI = 10

SYSTEM = """
Sei l'assistente di WiData
"""

def carica_storia():
    """Carica la history dal file JSON. Restituisce lista vuota se non esiste."""
    if os.path.exists(MEMORY_FILE):
        with open(MEMORY_FILE, "r", encoding="utf-8") as f:
            storia = json.load(f)
            print(f"📂 Storia caricata: {len(storia)} messaggi precedenti")
            return storia
    print("🆕 Nessuna storia precedente — nuova conversazione")
    return []

def salva_storia(history):
    """Salva la history su file JSON."""
    with open(MEMORY_FILE, "w", encoding="utf-8") as f:
        json.dump(history, f, ensure_ascii=False, indent=2)
    print(f"💾 Storia salvata: {len(history)} messaggi")

def chat_persistente(messaggio, history):
    """Chatbot con memoria persistente tra sessioni."""
    history.append({"role": "user", "content": messaggio})

    # Sliding Window
    if len(history) > MAX_MESSAGGI:
        history[:] = history[-MAX_MESSAGGI:]
        print(f"  ✂️  History troncata a {MAX_MESSAGGI} messaggi")

    params = {
        "model": "claude-haiku-4-5-20251001",
        "max_tokens": 400,
        "messages": history,
        "system": SYSTEM
    }
    full_text = ""
    print("WiData", end="", flush=True)

    # Streaming
    full_text = ""
    with client.messages.stream(**params) as stream:
        for text in stream.text_stream:
            print(text, end="", flush=True)
            # Stampa ogni token con print(text, end='', flush=True)
            full_text += text
        message = stream.get_final_message()
    history.append({"role": "assistant", "content": full_text})


    salva_storia(history)

    token_input = message.usage.input_tokens
    token_output = message.usage.output_tokens

    print(f"Token input: {token_input}, Token output: {token_output}, Token Totali: {token_input + token_output}")

    return full_text
# Loop principale
def main():
    history = carica_storia()
    print("🤖 Chatbot WiData avviato. Digita 'esci' per uscire.\n")

    while True:
        utente = input("Tu: ")
        if utente.lower() == "esci":
            print("👋 Arrivederci!")
            break
        risposta = chat_persistente(utente, history)
        # Lo streaming stampa già durante l'esecuzione

main()  # Decommentare per eseguire

🆕 Nessuna storia precedente — nuova conversazione
🤖 Chatbot WiData avviato. Digita 'esci' per uscire.

Tu: Ciao
WiData# Ciao! 👋

Sono l'assistente di **WiData**. 

Come posso aiutarti oggi? Sono a disposizione per:

- 📊 Rispondere a domande su **dati e analytics**
- 💻 Assistenza su **strumenti e piattaforme di data analysis**
- 📈 Supporto su **strategie data-driven**
- 🔍 Chiarimenti su **concetti di data science**
- E molto altro!

Dimmi pure di cosa hai bisogno! 😊💾 Storia salvata: 2 messaggi
Token input: 21, Token output: 142, Token Totali: 163
Tu: esci
👋 Arrivederci!


---
## 📤 Consegna

1. Completa tutti gli esercizi
2. Scarica: `File → Scarica → .ipynb`
3. Rinomina: `Lezione3_TUONOME.ipynb`
4. Carica su GitHub in `lezione3/`

```bash
git add lezione3/
git commit -m "Lezione 3 completata"
git push
```

---
### 📖 Per la prossima lezione (Giovedì 28/05)
Leggi **Huyen Cap. 6 — sezione RAG**

---
*ITS Novitas 4.0 — AI Engineering Fundamentals | Marco Uras*